# Future Plans

Ideas for improving Tigers and Goats training. This notebook describes future work; it does not run training or enable these features.

**Status:** Proposed | **Originally added:** 2026-10-19 | **Prioritized:** 2026-10-24

Ideas are ordered by expected usefulness and experimental interest for the current unmasked learner, balancing effort and dependencies. This is a judgment-based starting order, not a claim that an idea will improve results. Revisit it as diagnostics identify bottlenecks. Each idea remains subject to user confirmation before implementation.

Proposed configuration examples for opponent pools, sticky sampling, and randomized starts are not currently supported. Other ideas distinguish existing controls from future changes and experiments.

## Priority at a glance

Complexity describes development and testing effort, not model strength.

| Priority | Idea | Complexity | Why this order |
| --- | --- | --- | --- |
| 1 | Reward diagnostics and controlled experiments | Low to Medium | Highest immediate value: distinguish legality, game difficulty, and reward effects before adding more variables. |
| 2 | Reward and task curricula | Low for manual reward phases; higher for automatic progression | Strong next learning experiment: test whether staged guidance helps the policy turn promising positions into wins. |
| 3 | Blocking and near-lock shaping refinements | Medium to High | Directly targets the winning condition and the observed near-lock correlation, while checking misleading shaping incentives. |
| 4 | Neural-network skip connections | Medium | Interesting architecture experiment for learning efficiency; compare against the current MLP after fixing the reward/evaluation baseline. |
| 5 | Legal-only game turns and an attempt safety bound | Medium | Useful bounded experiment in episode rules, but earlier saved runs rarely reached the turn limit, so it may not fix the main bottleneck. |
| 6 | Direct randomized starting states | High | Could expose useful near-win and unfamiliar positions, but reliable state generation and bookkeeping require substantial work. |
| 7 | Zero-illegal episode bonus | Medium | Simple incentive experiment, but existing illegal penalties already provide feedback and a clean-episode bonus can favor short losses. |
| 8 | Multiple probabilistically mixed opponents | High | Valuable for later robustness once basic winning behavior is repeatable; loading and validating model pools adds complexity. |
| 9 | Consecutive-illegal timeout | Medium after Idea 5 | A follow-up if illegal streaks remain a measured problem; test after legal-turn and total-attempt rules to isolate its effect. |
| 10 | Sticky Markov opponent sampling | Medium after Idea 8 | Interesting opponent-sampling research, but depends on the model pool and is less directly tied to the current learning questions. |
| 11 | Model naming, embedded metadata, and experiment notes | Medium | Preserve experiment identity, training lineage, settings, and context when models are shared or resumed. |
| 12 | Complete TensorBoard window ending rates | Low | Account for every completed episode so outcome rates sum to 1, including illegal terminations. |

Keep each idea focused. Record completed work and experiment results in [Versions and progress](VERSIONS_AND_PROGRESS.ipynb).

## Future ideas from unmasked training

**Added:** 2026-10-24 | **Status:** Brainstormed; not approved for implementation.

These ideas record the reward, legality, and curriculum discussion. Confirm changes with the user before implementation. Values below are experimental starting points, not established best settings.

Already implemented: masked/unmasked controls, evaluation in both mask modes, illegal-action statistics and their renamed TensorBoard labels, reward-component tracking, and manual per-phase `reward_overrides`. The current PPO configuration already sets the Goat win reward to +6; raising it from +3.2 to +6 is not pending work. Do not confuse saved configuration with settings already loaded into a running kernel.

## 1. Reward diagnostics and controlled experiments

**Priority rationale:** Highest immediate value: distinguish legality, game difficulty, and reward effects before adding more variables.

**Goal:** Explain reward plateaus and rising illegal rates before changing several learning incentives. **Complexity: Low to Medium.**

Additional tracking proposed:

- Illegal attempts and rates split by placement versus movement, with sample counts. An increasing aggregate rate may reflect reaching harder positions rather than forgetting legality everywhere.
- Raw Tiger mobility, mobility changes, near-lock entries/exits, and conversion from near lock to win. Current `block_mean` and `near_lock_mean` are reward components, not raw counts.
- Episode reward, legal progress, illegal penalties, and length split by outcome. Record full-episode component totals to reconcile returns; current 1,000-step component means cannot be directly summed with differently scoped episode means.
- Matching recent windows for illegal penalty per completed episode and clean-episode percentage, alongside the existing cumulative metrics. Avoid comparing cumulative statistics with a recent window as though their denominators matched.
- Illegal streak distributions and safety-cutoff frequencies if Ideas 5 and 9 are implemented.

Controlled experiments proposed:

- Compare the existing +6 Goat win setting against a baseline or another modest win-reward value, keeping other terms fixed. Now that wins occur, a larger bonus can change their training weight, but it need not exceed the absolute magnitude of a losing return. Do not report +3.2 to +6 as unimplemented work.
- If illegal penalties dominate a specific stage, independently test a less-negative hard-invalid penalty, for example -1 to -0.25. It may ease exploration but also makes illegal attempts cheaper. Do not combine this with a win-reward or timeout change in the first comparison.
- Compare stochastic and deterministic unmasked evaluation, with an attempt safeguard. Consider a separate entropy-coefficient experiment: exploration, policy updates, and harder visited states can all increase illegal sampling; the curve alone does not establish profitable illegal behavior.
- Benchmark masked versus unmasked runs with matched architecture, seeds, training budgets, opponents, and evaluation rules. Account for legal actions completed and time, since timesteps count all attempts.

Greedy Tiger is stochastic but capture-biased: at capture bias 1 it always selects a capture when available, choosing randomly among captures; otherwise it chooses a random legal move. This may increase outcome variability, but it is not established as the primary training bottleneck.

**Interpretation:** Earlier saved runs near -5 had frequent capture losses and occasional wins. Six captures plus a -3.2 terminal loss contribute -5.3 before other terms; the currently saved -3 loss setting changes that subtotal to -5.1. Such a plateau alone does not demonstrate that illegal penalties dominate or that the policy found an optimum. Compare win rate on fixed held-out seed schedules; changing reward scale mechanically changes average reward.

## 2. Reward and task curricula

**Priority rationale:** Strong next learning experiment: test whether staged guidance helps the policy turn promising positions into wins.

**Goal:** Emphasize useful intermediate skills early, then emphasize converting them into wins. **Complexity: Low for manual reward phases; higher for automatic advancement or new starting-state tasks.**

Existing training phases already accept different `reward_overrides` while continuing the same model. What remains proposed is the curriculum design, experiments, and any automatic progression, not the basic phase API. Training phases are separate from the game's placement/movement phases.

Candidate stages:

1. Learn legal, useful play: keep illegal penalties and encourage Tiger restriction and Goat preservation.
2. Learn to close traps: maintain blocking shaping with modest near-lock encouragement.
3. Convert traps into wins: gradually reduce intermediate shaping while retaining stable win/loss and capture rewards.

First compare fixed rewards with a simple two-stage schedule: current blocking shaping, then reduced shaping after wins become repeatable. A manual schedule is easier to review initially. A future automatic schedule could use sustained held-out win rate and legality measures, with minimum evaluation counts and protection against switching phases on a noisy spike; thresholds are undecided.

Other proposed curricula: begin with an easier opponent or valid positions nearer a win, then broaden toward ordinary openings and harder opponents. Near-win starting states depend on the state-validation work in Idea 6. Existing Greedy/Smart mixing can support manual opponent schedules; arbitrary opponent pools remain separate work.

Keep final evaluation settings fixed across phases and methods. Changing rewards changes the meaning of training reward; compare win rate and legality across multiple seeds instead. Save the schedule and effective phase settings. Reward changes can temporarily disrupt the value function; measure transitions rather than assuming immediate improvement.

References: [Curriculum policies](https://arxiv.org/abs/1812.00285) and [RL curriculum framework](https://arxiv.org/abs/2003.04960).

## 3. Blocking and near-lock shaping refinements

**Priority rationale:** Directly targets the winning condition and the observed near-lock correlation, while checking misleading shaping incentives.

**Goal:** Encourage sustained progress toward immobilizing all Tigers while preserving enough Goats. **Complexity: Medium to High.**

Blocking currently rewards mobility reduction and penalizes increases; near lock means 1-2 Tiger options after the Goat action. Neither guarantees a win. The cached mobility comparison spans successive post-Goat snapshots, so it includes the intervening Tiger response rather than isolating only the latest Goat action.

Candidates to compare separately:

- Measure Tiger mobility immediately before and after the learner action, or across a full learner/opponent turn, with the intended attribution stated explicitly.
- Test an explicit penalty for losing a near lock. This does not exist separately today; mobility increases already incur a penalty, so avoid unintentionally double-penalizing them.
- Reward entering or improving a near lock rather than repeatedly paying for remaining there. Repeated bonuses could favor maintaining a promising position over finishing.
- Explore potential-based shaping using a board-quality score combining Tiger restriction and Goat safety. The shaping form is `gamma * Phi(next_state) - Phi(state)`, with consistent terminal handling. The theoretical policy-invariance result has conditions; it does not automatically apply to the current shaping or guarantee PPO convergence.

Do not assume that the strong correlation between near-lock reward and wins proves that increasing the bonus causes more wins. Partial blocking can expose captures or future escapes. Compare conversion from near lock to win and Goat survival.

Reference: [Policy invariance under reward transformations](https://people.eecs.berkeley.edu/~pabbeel/cs287-fa09/readings/NgHaradaRussell-shaping-ICML1999.pdf).

## 4. Neural-network skip connections

**Priority rationale:** Interesting architecture experiment for learning efficiency; compare against the current MLP after fixing the reward/evaluation baseline.

**Added:** 2026-10-24 | **Status:** Proposed; confirm before implementation. **Complexity: Medium.**

**Goal:** Test whether skip connections make neural-network training easier, faster, or both. These are hypotheses to measure, not guaranteed improvements.

Explore residual blocks within a single learner's policy/value network: a block learns a transformation `F(x)` and adds a skip path, for example `y = x + F(x)`. Start with matching feature widths; different widths require an explicit projection. This is a separate experiment from combining multiple saved models.

**Implementation needs:** Add a custom policy/network compatible with the notebook's MaskablePPO workflow, keeping action masking optional. Specify whether residual blocks apply to the actor, critic, or both; compare these separately. Preserve policy/value output shapes, device support, model saving/loading, and evaluation. Record the architecture and parameter count. Treat this as a new model architecture rather than assuming existing checkpoints can resume into it.

**Experiment:** Compare the current plain MLP with a residual variant using the same observations, rewards, opponents, optimizer settings, seed set, and evaluation rules. Keep widths/depth comparable and report parameter-count differences. Test masked and unmasked conditions separately. Do not change rewards or curricula in the same initial comparison.

**Success measures:** Training steps and wall-clock time to a predefined held-out win-rate target; illegal-action rate and zero-illegal episode percentage; stability across seeds; final win rate; and training/inference throughput. Distinguish fewer samples needed from faster execution: a model can learn in fewer steps while costing more time per step. Report runs that never reach the target.

**Tradeoffs and checks:** Extra computation and architectural complexity may outweigh learning gains, particularly for a shallow MLP. Check residual shapes, gradients, optional masking, and save/load prediction consistency before larger runs.

Related context: [Network architecture and model combiner](../docs/NETWORK_ARCHITECTURE_AND_MODEL_COMBINER.md). Skip paths between separate models are discussed there; this idea concerns residual connections inside an individual learner.

## 5. Legal-only game turns and an attempt safety bound

**Priority rationale:** Useful bounded experiment in episode rules, but earlier saved runs rarely reached the turn limit, so it may not fix the main bottleneck.

**Goal:** Give an unmasked learner the full legal-play budget without allowing an episode to retry illegal moves forever. **Complexity: Medium.**

Currently, illegal attempts consume turns toward `MAX_TURNS`. Proposed behavior: only legal learner actions increment the game-turn counter; illegal attempts still receive their existing penalties, leave the board and opponent idle, and consume training timesteps.

Track per-episode legal turns, illegal attempts, and total attempts separately. Example: 20 legal turns + 80 illegal attempts = 100 total attempts. Current callbacks already track action counts; proposed work exposes consistent per-episode environment counters and uses separate game and attempt budgets.

Test a generous total-attempt bound, for example 2,000 attempts with 100 legal turns. This guarantees resets without introducing a consecutive-illegal threshold in the first experiment. Use a distinct `AttemptLimit` outcome, keep it separate from ordinary maximum-turn timeouts, retain it in the win-rate denominator, and add no extra terminal penalty initially. Decide whether the cutoff is part of the task or an external interruption before selecting termination/truncation and value-bootstrapping behavior.

Keep reward values unchanged. Report how often the safety bound is reached: frequent cutoffs mean it materially changes training. Apply and record comparable rules in masked/unmasked experiments and evaluation. Deterministic unmasked evaluation can repeat the same illegal action on an unchanged observation, so evaluation also needs a bound. Training's finite timestep budget already bounds the overall run, but not individual episodes.

**Checks:** Both roles; unchanged legal-only trajectories; repeated illegal choices; counter resets in parallel environments; legal turn limits; attempt-bound outcomes; exported provenance. This counter change was discussed and reverted, not retained.

## 6. Direct randomized starting states

**Priority rationale:** Could expose useful near-win and unfamiliar positions, but reliable state generation and bookkeeping require substantial work.

**Goal:** Expose PPO to positions it rarely visits by sampling starting states directly, rather than only playing random warm-up moves from the normal opening.

**Complexity: High.** A valid state includes more than the visible board. Game counters, legal actions, and reward bookkeeping must agree with the position.

### Proposed setting — not currently supported

```python
"random_start_prob": 0.5,
```

At each episode reset, this means approximately 50% normal openings and 50% directly sampled starting states. Validate the setting as a finite probability from 0 to 1. A value of 0 preserves normal starts; 1 uses randomized starts for every training episode.

### Implementation needs

- Sample positions in both placement and movement phases, including piece locations and capture counts.
- Enforce game constraints: piece counts, phase, captures, and placement progress must agree.
- Reject invalid positions and positions where the game is already over.
- Initialize legal moves, turn counters, repetition tracking, and reward bookkeeping consistently with the starting scenario. Define the starting-history rules before implementation so the first reward and timeout behavior are meaningful.
- Use seeded randomness and bounded retries when rejecting candidates. Report failure clearly if no valid position can be generated within the retry limit.
- Support both learner roles and the selected scripted or model opponent.
- Save random-start settings and observed start statistics, including normal/random proportions and placement/movement proportions.

Positions can be unusual and do not need a demonstrated sequence of moves from the opening. However, passing basic validity checks does not prove a position is reachable in ordinary play. The goal is broader coverage of valid scenarios, not a claim that every generated position occurs in a normal game or that every state will be visited.

### Success checks and experiment

Check observations, piece counts, legal-action masks, and first-step reward bookkeeping. Test reproducibility, rejected candidates, retry limits, both learner roles, and both game phases. Confirm that `random_start_prob = 0` leaves normal reset behavior unchanged.

Compare normal-only training with mixed-start training using equal training-step budgets and multiple seeds. Report performance separately from normal openings and from held-out randomized positions. Also report generation overhead. Broader state coverage is useful only if the measured learning results support it.

## 7. Zero-illegal episode bonus

**Priority rationale:** Simple incentive experiment, but existing illegal penalties already provide feedback and a clean-episode bonus can favor short losses.

**Goal:** Reward completing an entire game without an illegal attempt. **Complexity: Medium.**

Test an optional +1 bonus, paid once at an ordinary game ending when the episode has zero illegal attempts. Include legal wins and legal losses; decide explicitly how existing game timeouts qualify. Exclude future safety/attempt-limit and illegal-streak cutoffs. Keep ordinary illegal penalties and log the bonus separately from the base terminal outcome and other shaping.

**Tradeoffs:** After the first illegal attempt the bonus is already lost, so it adds no further incentive against subsequent illegal attempts. It could favor short clean losses over longer games containing a mistake. Evaluate win rate and episode length as well as clean-episode percentage; a higher reward alone does not establish better strategy.

**Checks:** Award exactly once; any illegal attempt disqualifies the episode; independent counters in each parallel environment; reset handling; terminal rewards and bonus accounted for separately. This bonus is not currently implemented.

## 8. Multiple probabilistically mixed opponents

**Priority rationale:** Valuable for later robustness once basic winning behavior is repeatable; loading and validating model pools adds complexity.

**Goal:** Train one learner against a variety of scripted and saved opponents, rather than only two opponent types.

**Complexity: High.** This affects configuration, model loading and memory use, opponent selection, logging, and evaluation in both the notebook and training script.

### Proposed settings — not currently supported

Example for a goat learner; the saved file must contain a compatible Tiger policy:

```python
"opponent": ["greedy", "smart", "models/tiger_L2.zip"],
"mix_prob": [0.2, 0.3, 0.5],
```

The lists match by position: 20% Greedy Tiger, 30% Smart Tiger, and 50% the saved Tiger model. At the start of each game, select one opponent and keep it for the whole game. Actual proportions will vary, especially in short runs.

### Implementation needs

- Require nonempty lists of the same length.
- Require finite probabilities from 0 to 1. Their sum must equal 1 within a small floating-point tolerance; report invalid inputs clearly.
- Check scripted names for the learner's role: goats face Tiger opponents; tigers face Goat opponents. Check saved model files and compatibility before training.
- Keep opponent weights frozen. Load and reuse models efficiently, including within each training worker.
- Make sampling reproducible with a seed and preserve current string opponents and scalar mixing settings.
- Record opponent identities, requested probabilities, and observed selection frequencies.
- Provide matching training and evaluation support in the notebook and script. Evaluate against individual opponents as well as the mixture.

**Dependency:** Goats facing saved Tiger models need the loading and validation support described in the [Model Tiger training plan](../docs/MODEL_TIGER_TRAINING_PLAN.md). The current notebook already supports Tiger learners facing a saved Goat model, but not an arbitrary model pool.

### Success checks

Invalid list sizes, probabilities, names, and model files fail clearly. Every supported opponent works without changing its weights. Repeating a seed repeats the selection sequence. Across many games, observed selections approach the requested distribution. Existing configurations still work.

## 9. Consecutive-illegal timeout

**Priority rationale:** A follow-up if illegal streaks remain a measured problem; test after legal-turn and total-attempt rules to isolate its effect.

**Goal:** End an episode when the learner gets stuck repeatedly choosing illegal actions. **Complexity: Medium after Idea 5.**

Track an illegal streak per environment, increment it on an illegal attempt, and reset it after any legal action and at episode reset. Test a configurable cutoff such as 25 consecutive illegal attempts; allow disabling it. This threshold is a hypothesis, not a proven value.

Use a separate `IllegalTimeout` outcome, distinct from `AttemptLimit`, normal turn limits, wins, and losses. Preserve all attempt statistics. Initially retain normal illegal penalties without an additional cutoff penalty. Define termination/truncation semantics and bootstrapping explicitly before implementation.

**Tradeoff:** A policy could make one legal move just before the threshold, or seek an early exit from a bad game. Log streak distributions and check for clustering just below the limit. A total-attempt bound remains useful even with a streak cutoff.

**Experiment:** Test the legal-turn/total-attempt design first, then add the streak cutoff as a separate change. Compare win rate, legal progress, illegal rate, and cutoff frequency across seeds.

## 10. Sticky Markov opponent sampling

**Priority rationale:** Interesting opponent-sampling research, but depends on the model pool and is less directly tied to the current learning questions.

**Goal:** Let the learner play several consecutive games against the same opponent before drawing again from the pool.

**Complexity: Medium after Plan 8.** This adds selection history, seeded sampling, and logging on top of the opponent pool.

### Proposed setting — not currently supported

Add this alongside the opponent and probability lists from Plan 8:

```python
"stickiness": 0.8,
```

### Behavior

- The first game samples an opponent using the pool's probability list.
- Before each later game, keep the previous opponent with 80% probability. Otherwise, sample from the pool again.
- A new draw can select the same opponent, so 80% stickiness does not mean exactly a 20% chance of changing opponents.
- Change opponents only between games. Each environment keeps its own previous opponent.
- Start a fresh selection history at each curriculum phase.
- `stickiness = 0` means independent sampling each game. `stickiness = 1` keeps the initial opponent for the whole phase.

Validate stickiness as a finite number from 0 to 1. This controls the chance of staying, not a fixed number of games or training steps. The pool's probabilities describe game selections, not exact shares of training steps; longer games contribute more steps. With stickiness of 1, a single environment never explores the rest of its pool during that phase.

**Learning question:** Does spending longer with each opponent help adaptation, or does it encourage forgetting other opponents? Compare it against independent sampling with the same pool and training budget.

This changes how experience is collected. PPO uses a rollout buffer; this idea does **not** create a replay buffer or guarantee better learning.

### Success checks

Test stickiness of 0 and 1, invalid values, reproducible selections, separate histories across environments, and resets at phase boundaries. Log opponent streaks and observed frequencies. Verify that an opponent never changes mid-game.

## 11. Model naming, embedded metadata, and experiment notes

**Added:** 2026-10-28 | **Status:** Agreed design; proposed for future implementation. **Complexity: Medium.**

**Goal:** Keep model identity, opponent lineage, exact settings, and experiment intent available when a checkpoint is shared, renamed, evaluated, or resumed. This entry documents the design only; the features below are not yet implemented.

### Model naming

```text
L<level>_<G|T>vs<opponent>_v<project_version>[_seed<number>]_<steps>.zip
```

Examples:

```text
L1_GvsGT_v1.2_10M.zip
L1_GvsMixT_v1.2_seed43_20.5M.zip
L2_TvsModelG_v1.2_1.5M.zip
```

- `G` and `T` identify the learner. Opponent tags are `GT` (Greedy Tiger), `ST` (Smart Tiger), `MixT` (mixed Greedy/Smart Tigers), `ModelT` (saved Tiger), `RG` (Random Goat), and `ModelG` (saved Goat).
- Omit `_seed42`: an omitted seed denotes the default seed 42. Include other seeds explicitly. Always store the actual seed in metadata; embedded metadata remains authoritative for older files.
- The step suffix describes total learner training steps, including steps before a resume. It is a display label, not the exact count.
- Use uppercase `K` for thousands, `M` for millions, and `G` for billions. Select the largest applicable unit and round to the nearest 0.5 of that unit; exact halfway values round upward. Drop unnecessary `.0`: `10.1M` becomes `10M`, `10.3M` becomes `10.5M`, and `10.25M` becomes `10.5M`.
- Promote rounded `1000K` to `1M` and `1000M` to `1G`. Below 1,000 steps, show the integer count without a unit. Keep exact counts in metadata and retain collision-safe output folders; rounded names alone are not unique checkpoint identifiers.
- `model_name` is the base name without seed, step suffix, or extension: for example, `L1_GvsGT_v1.2`.

### Shared project version and metadata schema

- Use one shared `PROJECT_VERSION` across the notebook's environment, training, evaluation, and other modules. Use `MAJOR.MINOR`: increment minor for released updates and major for compatibility-breaking changes. Maintain one changelog covering all modules. Ordinary experiment-setting changes do not require a project-version bump.
- Store `project_version` in metadata. A newly trained stage uses the code version that produced it; preserve the parent checkpoint reference on resume.
- Independently store integer `schema_version`, initially `1`. Increment it when the metadata structure or field meanings change, not merely when code or reward values change.
- Existing models without project/schema versions remain legacy/unknown. Do not infer or retroactively assign versions from filenames.
- Scripted-opponent training is L1. Training against an Ln saved model produces L(n+1); multiple saved-opponent levels use the highest level plus one. Continuing against the same opponent keeps the level. Model level describes training lineage, not measured strength.

### Metadata contents and storage

Embed one structured metadata object in every periodic checkpoint and final model, and export matching readable JSON beside each archive. Refresh progress fields before each save. Save effective settings after loading any resumed model, not merely the requested configuration.

| Group | Fields and meaning |
| --- | --- |
| Versions | `schema_version`, `project_version` |
| Identity | `model_name`, `level`, `learner` (`goat` or `tiger`), `matchup` (for example, `GvsGT`) |
| Opponent | `opponent_family` uses the filename tag; `opponent_model` is the saved opponent's embedded model name; `opponent_checkpoint` identifies its particular archive. Both saved-opponent identifiers are `null` for scripted opponents. `mix_prob` records the actual mixture where applicable. |
| Experiment | `experiment_name`, `run_name`, `seed`, `use_masking`, `notes` |
| Progress and learner lineage | `parent_checkpoint`, `starting_timesteps`, `requested_additional_timesteps`, `total_timesteps`, `training_elapsed_seconds`. `parent_checkpoint` identifies the learner being resumed, not its opponent, and is `null` for fresh training. |
| Effective settings | `ppo` contains actual PPO hyperparameters, architecture, and activation; `environment` contains all effective reward and game settings, including defaults; `runtime` contains device, environment count, and relevant library versions. |
| Curriculum | `training_type`, ordered `phases`, and `current_phase` identifying the stage represented by the checkpoint |

Example identity for model-versus-model training (version values are illustrative):

```json
{
  "schema_version": 1,
  "project_version": "1.2",
  "model_name": "L2_TvsModelG_v1.2",
  "level": 2,
  "learner": "tiger",
  "matchup": "TvsModelG",
  "opponent_family": "ModelG",
  "opponent_model": "L1_GvsMixT_v1.1",
  "opponent_checkpoint": "L1_GvsMixT_v1.1_20M.zip",
  "seed": 42
}
```

### Curriculum history

Keep common PPO, environment, seed, and masking settings at the top level. Store an ordered `phases` list with each phase's number, name, status, matchup, opponent family/model/checkpoint, mix probability, reward overrides, requested/actual steps, starting/ending timestep counters, elapsed time, and output checkpoint. Results are `null` until known; update them and phase status during training. Each phase uses the shared environment settings plus its own overrides; omitted overrides do not inherit the previous phase's changes.

Example: one Goat model trains through Greedy Tiger, Smart Tiger, then mixed Tigers:

| Phase | Name | Matchup | Opponent family | Mix probability | Requested steps |
| --- | --- | --- | --- | --- | --- |
| 1 | greedy | `GvsGT` | `GT` | `null` | 2,000,000 |
| 2 | smart | `GvsST` | `ST` | `null` | 3,000,000 |
| 3 | mixed | `GvsMixT` | `MixT` | 0.5 | 5,000,000 |

Use overall `model_name = "L1_GvsMixT_v1.2"`, `matchup = "GvsMixT"`, `opponent_family = "MixT"`, and `training_type = "curriculum"`. The overall name summarizes the curriculum; phase records preserve the exact sequence. All opponents here are scripted, so the level stays L1. In phase 3, `mix_prob = 0.5` means 50% Smart Tiger and 50% Greedy Tiger, sampled per episode. Empty `reward_overrides` uses the shared settings. Exact total steps may exceed the requested budget because PPO completes rollouts.

### Experiment notes in the config cell

Proposed setting, not yet supported:

```python
EXPERIMENT_NOTES = """
Continue masking_test_02 for another 10M steps.
Keep the original reward weights and unmasked training.
Goal: measure whether more training improves the win rate
against Greedy Tiger.
"""
```

Save this string as `notes` in the embedded metadata and exported JSON. Use notes for intent, hypotheses, and context; retain rewards, seeds, opponents, and other settings in structured fields. Start with experiment-level notes; optional phase-specific notes can be considered later.

### Dependencies, boundaries, and success checks

- Model Tiger examples depend on the separate [Model Tiger training plan](../docs/MODEL_TIGER_TRAINING_PLAN.md). Metadata and names do not enable that training capability.
- This proposal does not implement training changes, rename existing models, modify saved archives, or start training. Preserve existing config-cell comments and examples when the feature is eventually implemented.
- Verify metadata survives model save/load, embedded and exported values agree, and every periodic/final save refreshes exact progress fields.
- Verify fresh, resumed, and multi-phase training preserve learner ancestry, phase history, opponent identity, effective settings, and notes. Check level calculation for scripted and saved opponents and preservation of `current_phase`.
- Verify legacy models remain loadable with unknown versions and missing optional metadata; do not fabricate historical values. Check default seed omission, explicit nondefault seeds, and embedded seed authority for older files.
- Check nearest-0.5 rounding, halfway-up behavior, uppercase units, values below 1,000, unit-promotion boundaries, exact metadata counters, and collisions between filenames that round to the same display label.

## 12. Complete TensorBoard window ending rates

**Added:** 2026-10-29 | **Status:** Requested; proposed for future implementation. **Complexity: Low.**

**Goal:** Show how every completed episode ends in the TensorBoard rolling-window statistics. Goat and Tiger win rates alone need not sum to 1: episodes can also end through illegal actions, time limits, repeat stalls, or other reasons.

### Proposed metrics and accounting

Use the same deque of completed episode outcomes and the same denominator for all ending rates: `N = len(window)`, up to the configured window size (currently 1,000 completed episodes). Before the window fills, divide by the number actually observed. Record one mutually exclusive outcome per completed episode, across all vector environments; do not count an unfinished episode or a nonterminal illegal action as an ending.

Retain the existing metrics and add the missing categories under `2.window_stats (per 1000 eps)/`:

| Metric | Outcome | Status |
| --- | --- | --- |
| `2.goat_win_rate` | Goat win | Already logged |
| `3.tiger_win_rate` | Tiger win | Already logged |
| `4.max_timeout_rate` | Turn-limit ending | Already logged |
| `5.repeat_timeout_rate` | Repeat-stall ending | Already logged |
| `6.illegal_termination_rate` | Episode ended by an illegal action | Proposed addition |
| `7.other_ending_rate` | Any remaining/unrecognized episode ending | Proposed addition |
| `8.ending_rate_sum` | Sum of the six outcome rates above | Proposed accounting check; should equal 1 |

For every nonempty window:

```text
goat_win_rate + tiger_win_rate
+ max_timeout_rate + repeat_timeout_rate
+ illegal_termination_rate + other_ending_rate = 1
```

- Keep `1.episodes_tracked` as the sample count. Publish the full set together at the existing logging cadence. With no completed episodes, omit rates rather than imply that an empty window accounts for 100% of endings.
- Here, "termination rate" means the specific illegal-action ending category, not every Gymnasium `terminated=True` event: wins are already counted separately. Illegal endings should be recognized by the classified outcome, including legacy illegal endings reported as truncations.
- Reuse the existing episode-outcome classification. An illegal attempt that continues play is not an illegal termination. If an illegal attempt reaches the turn limit and is classified as `MaxTimeout`, count it only as a max timeout.
- Keep the attempted-action illegal rate separate: its denominator is action attempts, whereas `illegal_termination_rate` uses completed episodes. Do not include action rates or overlapping summaries in the six-rate sum.
- Preserve the `Other` bucket so unfamiliar ending reasons cannot silently disappear. Nonzero `other_ending_rate` should prompt inspection of those reasons. Future new ending categories must partition this bucket rather than double-count episodes.

### Success checks and boundaries

- Feed a window containing all six outcomes and verify exact counts, a common denominator, and a rate sum of 1 within floating-point tolerance.
- Check partial windows, full windows, eviction of old episodes, and several environments ending on the same callback step. The sum must remain 1 whenever at least one completed episode is tracked.
- Check all-illegal endings, zero-illegal endings, unrecognized endings, nonterminal illegal attempts, and an illegal attempt ending at the turn limit. Verify each completed episode enters exactly one category.
- Confirm TensorBoard publishes all six rates and the sum at the same step, while retaining existing win/timeout tags, sample counts, and opponent-mix metrics.
- Documentation only for now: this proposal does not change callbacks, rewards, episode rules, or saved models. These are rolling-window fractions in the range 0 to 1, not percentages or cumulative rates.

## Related plans and references

- [Model Tiger training plan](../docs/MODEL_TIGER_TRAINING_PLAN.md): support for Goat learners facing frozen saved Tiger policies.
- [Network architecture and model combiner](../docs/NETWORK_ARCHITECTURE_AND_MODEL_COMBINER.md): current policy structure and an idea for combining trained models.
- [Versions and progress](VERSIONS_AND_PROGRESS.ipynb): completed changes, results, and next steps.

These are ideas and supporting references, not promises that the features already exist.

## Template for a new idea

Copy this into a new Markdown cell above the template:

### [Idea title]

**Status:** Proposed  
**Goal:** [What should improve, and for whom?]  
**Complexity:** Low / Medium / High — [Why?]

**Proposed settings — not currently supported:** [Optional example.]

**Behavior and implementation needs:** [How it should work and what must change.]

**Dependencies:** [Earlier features or documents needed first.]

**Success checks:** [How to verify correct behavior and measure whether the idea helps.]